In [1]:
import os, sys, socket, torch, pathlib, shutil, subprocess, textwrap
print("cwd:", os.getcwd())
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("content exists:", pathlib.Path("/content").exists())
print("local project exists:", pathlib.Path("/home/yaroslav").exists())
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA version :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CPU cores visible:", os.cpu_count())

cwd: /content
host: ab207eabbc5f
cuda: True
content exists: True
local project exists: False
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA version : 12.8
CUDA available: True
GPU name: Tesla T4
CPU cores visible: 2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SRC = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs"
DST = "/content/DUMPLINGs"

os.makedirs(DST, exist_ok=True)
os.chdir(SRC)
! git checkout -- config.json
! git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 340 bytes | 7.00 KiB/s, done.
From https://github.com/BlackSabbitch/DUMPLINGs
   ec4a8cf..ef6cb5e  main       -> origin/main
Updating ec4a8cf..ef6cb5e
Fast-forward
 config.json | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [4]:
LOCAL_RUNS = "/content/DUMPLINGs/runs"
DRIVE_RUNS = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

if os.path.exists(DST):
    shutil.rmtree(DST)

for name in [
    "run.py", "trainer.py", "evaluator.py", "extractor.py", "splitter.py",
    "tokenizer.py", "utils.py", "logger.py", "loss_functions.py",
    "config.json", "requirements.txt", "run.sh", "README.md",
    "bad_complexes.toml",
    "models", "parsers", "scripts"
]:
    s = os.path.join(SRC, name)
    d = os.path.join(DST, name)
    if os.path.isdir(s):
        shutil.copytree(s, d)
    elif os.path.exists(s):
        shutil.copy2(s, d)

# ESM cache is optional, but if present we reuse it
esm_cache_src = os.path.join(SRC, "esm_cache")
esm_cache_dst = os.path.join(DST, "esm_cache")
os.makedirs(esm_cache_dst, exist_ok=True)

if os.path.exists(esm_cache_src):
    for name in os.listdir(esm_cache_src):
        s = os.path.join(esm_cache_src, name)
        d = os.path.join(esm_cache_dst, name)
        if os.path.isdir(s):
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print(f"Copied ESM cache from {esm_cache_src}")
else:
    print("No ESM cache found on Drive; empty esm_cache/ will be used.")

shutil.copy2(
    "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/pdbbind_v2016.tar.gz",
    "/content/DUMPLINGs/pdbbind_v2016.tar.gz"
)

os.chdir(DST)
print(os.getcwd())

/content/DUMPLINGs


In [5]:
# Install non-PyG requirements only
skip_prefixes = (
    "torch",
    "torch-geometric",
    "torch-scatter",
    "torch-sparse",
    "torch-cluster",
    "pyg-lib",
)

reqs = []
with open("requirements.txt") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith(skip_prefixes):
            continue
        reqs.append(line)

print("Installing filtered requirements:")
for r in reqs:
    print(" ", r)

import subprocess
subprocess.run(["pip", "install", *reqs], check=True)

Installing filtered requirements:
  rdkit
  biopython
  pandas
  tqdm
  ipywidgets
  matplotlib
  pennylane
  uniplot


CompletedProcess(args=['pip', 'install', 'rdkit', 'biopython', 'pandas', 'tqdm', 'ipywidgets', 'matplotlib', 'pennylane', 'uniplot'], returncode=0)

In [6]:
# Colab environment bootstrap for DUMPLINGs
def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Make matplotlib and temp writes local to the VM, not Drive
os.environ["MPLCONFIGDIR"] = "/content/.mplconfig"
os.environ["TMPDIR"] = "/content/.tmp"
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Derive the PyG wheel index from the active torch build
torch_base = torch.__version__.split("+")[0]
cuda_tag = f"cu{torch.version.cuda.replace('.', '')}" if torch.version.cuda else "cpu"
pyg_url = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
print("PyG wheel index:", pyg_url)

# Remove possibly incompatible installs
run("pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric || true")

# Install the compiled extensions first, then torch-geometric
run(f"pip install pyg-lib torch-scatter torch-sparse torch-cluster -f {pyg_url}")
run("pip install torch-geometric")

# Verify CUDA support in torch_sparse
import torch_sparse
print("torch_sparse:", torch_sparse.__file__)

row = torch.tensor([0, 1], device="cuda")
col = torch.tensor([1, 0], device="cuda")
sp = torch_sparse.SparseTensor(row=row, col=col, sparse_sizes=(2, 2))
print("torch_sparse CUDA check OK:", sp.device())

print(textwrap.dedent("""
Recommended Colab workflow:
1. Keep the repo and training outputs in /content while running.
2. Copy final runs/ and cached artifacts back to Drive after training.
3. Avoid heavy read/write loops directly on mounted Google Drive.
"""))

PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric || true
$ pip install pyg-lib torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ pip install torch-geometric
torch_sparse: /usr/local/lib/python3.12/dist-packages/torch_sparse/__init__.py
torch_sparse CUDA check OK: cuda:0

Recommended Colab workflow:
1. Keep the repo and training outputs in /content while running.
2. Copy final runs/ and cached artifacts back to Drive after training.
3. Avoid heavy read/write loops directly on mounted Google Drive.



In [7]:
sync_cmd = f"""
while true; do
  rsync -a --delete "{LOCAL_RUNS}/" "{DRIVE_RUNS}/"
  sleep 120
done
"""

sync_proc = subprocess.Popen(["bash", "-lc", sync_cmd])
print("Sync PID:", sync_proc.pid)

Sync PID: 13948


In [8]:
! chmod +x run.sh
! ./run.sh --extract

[INFO][EXPERIMENT] Starting experiment: DUMPLING_A1b_DimeNet_Initial_milestone_bs2
[INFO][EXPERIMENT] Experiment signature: DUMPLING_A1b_DimeNet_Initial_milestone_bs2_20260503_210815
[INFO][EXPERIMENT] Base Datasets folder: datasets
[INFO][EXPERIMENT] Run results folder: runs/DUMPLING_A1b_DimeNet_Initial_milestone_bs2_20260503_210815
[INFO][EXPERIMENT] Log file: runs/DUMPLING_A1b_DimeNet_Initial_milestone_bs2_20260503_210815/log.txt
[INFO][InteractionGraphParser] Initialized dist_threshold=5.0, ca_only=False
[INFO][REGISTRY] Loaded bad complexes registry from bad_complexes.toml with 1 entries
[INFO][EXTRACTION] Unpacking 4057 complexes...
Extracting refined: 66444file [00:55, 1192.07file/s]
[INFO][EXTRACTION] Unpacking completed.
[WARNING][REGISTRY] Excluded 1 complexes from subset refined using bad_complexes.toml: 4bps(training)
[INFO][BUILD] Starting parallel parsing on 2 cores...
100%|██████████| 4056/4056 [01:08<00:00, 59.00it/s]
[INFO][BUILD] Success: 4056, Errors: 0
[INFO][SAVE] 

In [9]:
sync_proc.terminate()

In [10]:
! du -sh /content/DUMPLINGs/esm_cache 2>/dev/null || echo "esm_cache is absent"
! find /content/DUMPLINGs/esm_cache -type f | wc -l
! rsync -a "/content/DUMPLINGs/runs/" "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs/"
! rsync -a --delete "/content/DUMPLINGs/esm_cache/" "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/esm_cache/"